In [1]:
import gymnasium as gym
from gymnasium import spaces
from gymnasium.wrappers import TimeLimit
import numpy as np
from scipy.integrate import solve_ivp 
import os
from os import path
from collections import defaultdict
import time

class UnbalancedDisk(gym.Env):
    def __init__(self, umax=20., dt = 0.025, render_mode='human'):
        self.omega0 = 11.339846957335382
        self.delta_th = 0
        self.gamma = 1.3328339309394384
        self.Ku = 28.136158407237073
        self.Fc = 6.062729509386865
        self.coulomb_omega = 0.001

        self.umax = umax
        self.dt = dt 

        self.action_space = spaces.Box(low=-umax,high=umax,shape=tuple()) 
        low = [-float('inf'),-40] 
        high = [float('inf'),40]
        self.observation_space = spaces.Box(low=np.array(low,dtype=np.float32),high=np.array(high,dtype=np.float32),shape=(2,))

        # A simple cosine reward: highest at the top (pi), lowest at the bottom (0).
        # We also subtract a tiny penalty for high angular velocity.
        self.reward_fun = lambda self: -(np.cos(self.th) + 1) - 0.01 * (self.omega ** 2)        
        
        self.render_mode = render_mode
        self.viewer = None
        self.u = 0 
        self.reset()

    def step(self, action):
        self.u = action 
        self.u = np.clip(self.u,-self.umax,self.umax)
        
        def f(t,y):
            th, omega = y
            dthdt = omega
            friction = self.gamma*omega + self.Fc*np.tanh(omega/self.coulomb_omega)
            domegadt = -self.omega0**2*np.sin(th+self.delta_th) - friction + self.Ku*self.u
            return np.array([dthdt, domegadt])
            
        sol = solve_ivp(f,[0,self.dt],[self.th,self.omega]) 
        self.th, self.omega = sol.y[:,-1]

        reward = self.reward_fun(self)
        return self.get_obs(), reward, False, False, {}
         
    def reset(self, seed=None, options=None):
        # We also need to call super() to properly handle the seed in Gymnasium
        super().reset(seed=seed) 
        
        self.th = np.random.normal(loc=0,scale=0.001)
        self.omega = np.random.normal(loc=0,scale=0.001)
        self.u = 0
        return self.get_obs(), {}

    def get_obs(self):
        self.th_noise = self.th + np.random.normal(loc=0,scale=0.001) 
        self.omega_noise = self.omega + np.random.normal(loc=0,scale=0.001) 
        return np.array([self.th_noise, self.omega_noise])

    def render(self):
        import pygame
        from pygame import gfxdraw
        
        screen_width = 500
        screen_height = 500

        th = self.th
        omega = self.omega 

        if self.viewer is None:
            pygame.init()
            pygame.display.init()
            self.viewer = pygame.display.set_mode((screen_width, screen_height))

        self.surf = pygame.Surface((screen_width, screen_height))
        self.surf.fill((255, 255, 255))
        
        gfxdraw.filled_circle(self.surf, screen_width//2, screen_height//2, int(screen_width/2*0.65*1.3), (32,60,92))
        gfxdraw.filled_circle(self.surf, screen_width//2, screen_height//2, int(screen_width/2*0.06*1.3), (132,132,126))
        
        from math import cos, sin
        r = screen_width//2*0.40*1.3
        gfxdraw.filled_circle(self.surf, int(screen_width//2-sin(th)*r), int(screen_height//2-cos(th)*r), int(screen_width/2*0.22*1.3), (155,140,108))
        gfxdraw.filled_circle(self.surf, int(screen_width//2-sin(th)*r), int(screen_height//2-cos(th)*r), int(screen_width/2*0.22/8*1.3), (71,63,48))
        
        # os.getcwd() gets the notebook's current working directory safely
        fname = os.path.join(os.getcwd(), "clockwise.png")
        try:
            self.arrow = pygame.image.load(fname)
            if self.u:
                if isinstance(self.u, (np.ndarray,list)):
                    if self.u.ndim==1:
                        u = self.u[0]
                    elif self.u.ndim==0:
                        u = self.u
                    else:
                        raise ValueError(f'u={u} is not the correct shape')
                else:
                    u = self.u
                arrow_size = abs(float(u)/self.umax*screen_height)*0.25
                Z = (arrow_size, arrow_size)
                arrow_rot = pygame.transform.scale(self.arrow,Z)
                if self.u<0:
                    arrow_rot = pygame.transform.flip(arrow_rot, True, False)
                    
            self.surf = pygame.transform.flip(self.surf, False, True)
            self.viewer.blit(self.surf, (0, 0))
            if self.u:
                self.viewer.blit(arrow_rot, (screen_width//2-arrow_size//2, screen_height//2-arrow_size//2))
        except FileNotFoundError:
            self.surf = pygame.transform.flip(self.surf, False, True)
            self.viewer.blit(self.surf, (0, 0))
            
        if self.render_mode == "human":
            pygame.event.pump()
            pygame.display.flip()

        return True

    def close(self):
        if self.viewer is not None:
            import pygame
            pygame.display.quit()
            pygame.quit()
            self.isopen = False
            self.viewer = None

class UnbalancedDisk_sincos(UnbalancedDisk):
    def __init__(self, umax=3., dt = 0.025):
        super(UnbalancedDisk_sincos, self).__init__(umax=umax, dt=dt)
        low = [-1,-1,-40.] 
        high = [1,1,40.]
        self.observation_space = spaces.Box(low=np.array(low,dtype=np.float32),high=np.array(high,dtype=np.float32),shape=(3,))

    def get_obs(self):
        self.th_noise = self.th + np.random.normal(loc=0,scale=0.001) 
        self.omega_noise = self.omega + np.random.normal(loc=0,scale=0.001) 
        return np.array([np.sin(self.th_noise), np.cos(self.th_noise), self.omega_noise])

In [ ]:
class Discretize_obs(gym.ObservationWrapper):
    def __init__(self, env, nvec=10):
        super().__init__(env)
        if isinstance(nvec, int):
            self.nvec_array = np.array([nvec] * np.prod(env.observation_space.shape, dtype=int))
        else:
            self.nvec_array = np.array(nvec)
            
        self.observation_space = gym.spaces.MultiDiscrete(self.nvec_array)
        self.olow = env.observation_space.low
        self.ohigh = env.observation_space.high

    def observation(self, obs):
        scaled = (obs - self.olow) / (self.ohigh - self.olow) * self.nvec_array
        discretized = np.clip(scaled, 0, self.nvec_array - 1).astype(int)
        return tuple(discretized)

class DiscretizeAction(gym.ActionWrapper):
    def __init__(self, env, bins=5):
        super().__init__(env)
        self.bins = bins
        self.action_space = gym.spaces.Discrete(self.bins)
        
        low = np.atleast_1d(env.action_space.low)[0]
        high = np.atleast_1d(env.action_space.high)[0]
        
        self.action_mapping = np.linspace(low, high, self.bins)

    def action(self, act):
        # Cast the selected mapping directly to a float (scalar)
        return float(self.action_mapping[act])

def argmax(a):
    a = np.array(a)
    return np.random.choice(np.arange(len(a), dtype=int)[a == np.max(a)])

def Qlearn(env, nsteps=5000, alpha=0.2, eps_start=1.0, eps_end=0.01, gamma=0.99):
    Qmat = defaultdict(float) 
    
    env_time = env
    while hasattr(env_time, 'env') and not isinstance(env_time, gym.wrappers.TimeLimit):
        env_time = env_time.env
        
    ep_lengths = []
    ep_lengths_steps = []
    
    obs, info = env.reset()
    print(f'Training started for {nsteps} steps...')
    
    for z in range(nsteps):
        # --- EPSILON DECAY ---
        # Linearly decay from eps_start down to eps_end
        eps = eps_start - (eps_start - eps_end) * (z / nsteps)
        
        if np.random.uniform() < eps:
            action = env.action_space.sample()
        else:
            action = argmax([Qmat[obs, i] for i in range(env.action_space.n)])

        obs_new, reward, terminated, truncated, info = env.step(action)

        if terminated: 
            ep_lengths.append(getattr(env_time, '_elapsed_steps', 0))
            ep_lengths_steps.append(z)
            A = reward - Qmat[obs, action] 
            Qmat[obs, action] += alpha * A
            obs, info = env.reset()
        else: 
            max_q_next = max(Qmat[obs_new, a] for a in range(env.action_space.n))
            A = reward + gamma * max_q_next - Qmat[obs, action]
            Qmat[obs, action] += alpha * A
            obs = obs_new
            
            if truncated: 
                ep_lengths.append(getattr(env_time, '_elapsed_steps', 0))
                ep_lengths_steps.append(z)
                obs, info = env.reset()
                
    print('Training complete.')
    return Qmat, np.array(ep_lengths_steps), np.array(ep_lengths)

def save_qmat(Qmat, filename):
    rows = []

    for (obs, action), q_value in Qmat.items():
        # obs is something like (theta_bin, omega_bin, ...)
        row = list(obs) + [action, q_value]
        rows.append(row)

    rows = np.array(rows, dtype=float)
    np.savez_compressed(filename, rows=rows)


def load_qmat(filename, obs_dim):
    data = np.load(filename)
    rows = data["rows"]

    Qmat = defaultdict(float)

    for row in rows:
        obs = tuple(row[:obs_dim].astype(int))
        action = int(row[obs_dim])
        q_value = float(row[obs_dim + 1])

        Qmat[obs, action] = q_value

    return Qmat

### Load existing model, otherwise train model

In [20]:
from collections import defaultdict
from pathlib import Path
# 1. Initialize the sincos environment to bound the angle observations
env = UnbalancedDisk_sincos(umax=3.0, dt=0.025)

# 2. Add a TimeLimit so the agent experiences resets (200 steps = 5 seconds)
env = TimeLimit(env, max_episode_steps=200)

# 3. Discretize spaces 
env = Discretize_obs(env, nvec=[15, 15, 30]) 
env = DiscretizeAction(env, bins=7)

# 4. Train the Agent 
# (likely need ~50,000+ steps for it to balance well)
qmat_path = Path("Qmat_unbalanced_disk.npz")

# Number of observation variables
obs_dim = 3   # since nvec=[15, 15, 30]
#If model exists, load it, else train new
if qmat_path.exists():
    print("Loading saved Qmat...")
    Qmat = load_qmat(qmat_path, obs_dim)

    steps = None
    lengths = None
else:
    print("No saved Qmat found, training...")
    Qmat, steps, lengths = Qlearn(env, nsteps=1000000)

    save_qmat(Qmat, qmat_path)
    print("Qmat saved.")


# 5. Test and Render the Trained Policy
print("Testing the trained policy...")
obs, info = env.reset()

try:
    for i in range(10000):
        action = argmax([Qmat[obs, a] for a in range(env.action_space.n)])
        obs, reward, terminated, truncated, info = env.step(action)
        
        env.render()
        time.sleep(0.025) # Sleep to match the dt of the physics engine
        
        if terminated or truncated:
            print("Test episode finished!")
            break
finally:
    env.close() # Closes Pygame safely to save your Jupyter Kernel!
#Save the Q-matrix for later use



Loading saved Qmat...
Testing the trained policy...
Test episode finished!


In [12]:
# 5. Test and Render the Trained Policy
print("Testing the trained policy...")
obs, info = env.reset()
print(Qmat)
try:
    for i in range(1000000):
        action = argmax([Qmat[obs, a] for a in range(env.action_space.n)])
        obs, reward, terminated, truncated, info = env.step(action)
        
        env.render()
        time.sleep(0.025) # Sleep to match the dt of the physics engine
        
        if terminated or truncated:
            print("Test episode finished!")
            break
finally:
    env.close() # Closes Pygame safely to save your Jupyter Kernel!

Testing the trained policy...
defaultdict(<class 'float'>, {((np.int64(7), np.int64(14), np.int64(14)), 0): -50.08031185034713, ((np.int64(7), np.int64(14), np.int64(14)), 1): -52.22488623530227, ((np.int64(7), np.int64(14), np.int64(14)), 2): -52.169140994119346, ((np.int64(7), np.int64(14), np.int64(14)), 3): -52.37593821207147, ((np.int64(7), np.int64(14), np.int64(14)), 4): -51.90486600841737, ((np.int64(7), np.int64(14), np.int64(14)), 5): -52.05003063215615, ((np.int64(7), np.int64(14), np.int64(14)), 6): -52.149522852640395, ((np.int64(7), np.int64(14), np.int64(15)), 0): -52.37044772874577, ((np.int64(6), np.int64(14), np.int64(13)), 0): -47.63236017972194, ((np.int64(6), np.int64(14), np.int64(13)), 1): -48.960432795731855, ((np.int64(6), np.int64(14), np.int64(13)), 2): -49.23345191191639, ((np.int64(6), np.int64(14), np.int64(13)), 3): -48.971815428444046, ((np.int64(6), np.int64(14), np.int64(13)), 4): -49.12938178834356, ((np.int64(6), np.int64(14), np.int64(13)), 5): -49.